# Can we predict whether a warranty claim is filed within a year?
Target: `claim_within_year` (1 = claim within a year of sale). Plan: inspect data, check signal per feature, compare models against a naive baseline using cross-validation, then confirm on a held-out test set.

`device_serial` is an identifier, so it is **excluded** from the features.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
SEED = 42
df = pd.read_csv('data.csv')
print(df.shape); print(df.isna().sum().sum(), 'missing;', df.device_serial.duplicated().sum(), 'duplicate serials')
df.describe(include='all').T

## Target balance and feature signal

In [ ]:
y_rate = df.claim_within_year.mean(); print(f'Claim rate: {y_rate:.3f}')
print(df.groupby('channel').claim_within_year.agg(['mean', 'size']).round(3))
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for a, col in zip(ax, ['device_age_years', 'price_eur', 'daily_hours']):
    g = df.groupby(pd.qcut(df[col], 10, duplicates='drop'), observed=True).claim_within_year.mean()
    a.plot(range(len(g)), g.values, marker='o'); a.axhline(y_rate, ls='--', c='grey')
    a.set_title(f'Claim rate by {col} decile'); a.set_xlabel('decile')
plt.tight_layout(); plt.show()
df[['device_age_years', 'price_eur', 'daily_hours']].describe().loc[['min', '50%', 'max']]

Claim rate rises with age, price and daily hours; the partner channel is somewhat higher. Price and daily hours are right-skewed, so the linear model gets log-transformed versions; the tree model does not need that.

In [ ]:
num = ['device_age_years', 'price_eur', 'daily_hours']; cat = ['channel']
X = df[num + cat]; y = df.claim_within_year
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
logt = FunctionTransformer(np.log1p, feature_names_out='one-to-one')
lin_prep = ColumnTransformer([
    ('age', StandardScaler(), ['device_age_years']),
    ('skew', make_pipeline(logt, StandardScaler()), ['price_eur', 'daily_hours']),
    ('cat', OneHotEncoder(drop='first'), cat)])
models = {
    'baseline (prior)': DummyClassifier(strategy='prior'),
    'logistic': make_pipeline(lin_prep, LogisticRegression(max_iter=1000)),
    'gradient boosting': make_pipeline(ColumnTransformer([('c', OneHotEncoder(), cat)], remainder='passthrough'),
        HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, l2_regularization=1.0, random_state=SEED)),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = {}
for n, m in models.items():
    r = cross_validate(m, Xtr, ytr, cv=cv, scoring=['roc_auc', 'average_precision', 'neg_brier_score'])
    rows[n] = {'ROC AUC': r['test_roc_auc'].mean(), 'AUC std': r['test_roc_auc'].std(),
               'PR AUC': r['test_average_precision'].mean(), 'Brier': -r['test_neg_brier_score'].mean()}
pd.DataFrame(rows).T.round(3)

Cross-validation uses the training split only (80%). PR AUC should be judged against the claim rate (about 0.126), which is what a no-skill model scores.

In [ ]:
test = {}
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for n, m in models.items():
    m.fit(Xtr, ytr); p = m.predict_proba(Xte)[:, 1]
    test[n] = {'ROC AUC': roc_auc_score(yte, p), 'PR AUC': average_precision_score(yte, p), 'Brier': brier_score_loss(yte, p)}
    if n.startswith('baseline'): continue
    fpr, tpr, _ = roc_curve(yte, p); ax[0].plot(fpr, tpr, label=n)
    pr, rc, _ = precision_recall_curve(yte, p); ax[1].plot(rc, pr, label=n)
    fp, mp = calibration_curve(yte, p, n_bins=8, strategy='quantile'); ax[2].plot(mp, fp, marker='o', label=n)
ax[0].plot([0, 1], [0, 1], 'k:'); ax[0].set(title='ROC', xlabel='FPR', ylabel='TPR')
ax[1].axhline(yte.mean(), c='k', ls=':'); ax[1].set(title='Precision-recall', xlabel='recall', ylabel='precision')
ax[2].plot([0, .5], [0, .5], 'k:'); ax[2].set(title='Calibration', xlabel='predicted', ylabel='observed')
ax[0].legend(); plt.tight_layout(); plt.show()
pd.DataFrame(test).T.round(3)

## What drives the predictions, and how useful are they?

In [ ]:
best = models['logistic']; p = best.predict_proba(Xte)[:, 1]
pi = permutation_importance(best, Xte, yte, scoring='roc_auc', n_repeats=20, random_state=SEED)
print(pd.Series(pi.importances_mean, index=Xte.columns).sort_values(ascending=False).round(4))
t = pd.DataFrame({'p': p, 'y': yte.values}); t['decile'] = pd.qcut(t.p.rank(method='first'), 10, labels=range(1, 11))
t.groupby('decile', observed=True).y.agg(['mean', 'size']).rename(columns={'mean': 'claim_rate'}).round(3)

In [ ]:
# Bootstrap CI on test ROC AUC for the logistic model
rng = np.random.default_rng(SEED); a = []; yv = yte.values
for _ in range(1000):
    i = rng.integers(0, len(yv), len(yv)); a.append(roc_auc_score(yv[i], p[i]))
print('Logistic test ROC AUC 95% CI:', np.percentile(a, [2.5, 97.5]).round(3))

## Conclusions
(Numbers from one run with seed 42; re-run to refresh.)

- **Yes, partially.** Logistic regression reaches ROC AUC about 0.73 on the held-out 20% (95% bootstrap CI about 0.69-0.77) and PR AUC about 0.32, against 0.50 and 0.126 for a no-skill model. CV (0.725 +/- 0.01) agrees with the test score.
- **Useful for ranking, not for confident individual calls.** The top-decile of predicted risk has a claim rate of about 34% (2.7x the 12.6% base rate); the bottom three deciles are around 3-5%. Most claims still sit in mid-risk devices.
- **Drivers:** device age, daily hours and price (older, heavier use, pricier means more claims); sales channel adds little beyond a slightly higher partner rate.
- **Simple beats complex here:** gradient boosting did no better than logistic regression, which suggests the signal is mostly smooth and monotone with the few features available. Predictions are reasonably calibrated (Brier 0.100 vs 0.110 baseline).
- **Caveats:** only four features, no claim timing or product category; the 12.6% positive rate means thresholds should be chosen from the business cost of false alarms vs missed claims, not accuracy.